### Codeforces Diagnostic Dataset 정제 기록
- reference : https://huggingface.co/collections/open-r1/codeforces?utm_source=chatgpt.com
- source : open-r1/codeforces

1. Codeforces 데이터셋을 추가한 목적   
- LiveCodeBench v6를 주 benchmark로 사용하고, 알고리즘 유형별 planning 능력의 차이를 진단하기 위한 별도의 Codeforces diagnostic dataset을 구축하였다.

2. CodeForces 데이터셋의 문제점
- open-r1/codeforces 데이터셋의 test split은 총 468개 문제로 구성되어 있었으나, 실제 코드 생성 평가에 사용할 공식 테스트 케이스가 불완전하게 수집된 경우를 확인했다. 따라서 official_tests_complete = True이고 2023,2024년에 수집된 데이터셋으로 필터링하여 20개 문제로 추려졌다.
- train split을 추가 활용하기로 했으며, 위와 같은 필터링 기준으로 78개 문제를 확보했다.

3. 최종 Codeforces Diagnostic Dataset
- 98 problems
- 28 algorithmic tags(불균형 존재)

#### 1. Dataset 기본 구조

In [1]:
from datasets import load_dataset


HF_CACHE = "/mnt/hdd/hf_cache"


# ============================================================
# Load Codeforces
# ============================================================

codeforces_train = load_dataset(
    "open-r1/codeforces",
    name="default",
    split="train",          # test
    cache_dir=HF_CACHE,
)

codeforces_test = load_dataset(
    "open-r1/codeforces",
    name="default",
    split="test",          # test
    cache_dir=HF_CACHE,
)


# ============================================================
# Basic Dataset Structure
# ============================================================

print("=" * 70)
print("Codeforces Dataset")
print("=" * 70)

print(f"[Train set] Number of samples : {len(codeforces_train)}")
print(f"[Train set] Columns {len(codeforces_train.column_names)}           : {codeforces_train.column_names}")

print(f"[Test set] Number of samples : {len(codeforces_test)}")
print(f"[Test set] Columns {len(codeforces_test.column_names)}           : {codeforces_test.column_names}")

/mnt/hdd/conda_envs/slm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Codeforces Dataset
[Train set] Number of samples : 9556
[Train set] Columns 27           : ['id', 'aliases', 'contest_id', 'contest_name', 'contest_type', 'contest_start', 'contest_start_year', 'index', 'time_limit', 'memory_limit', 'title', 'description', 'input_format', 'output_format', 'interaction_format', 'note', 'examples', 'editorial', 'rating', 'tags', 'testset_size', 'official_tests', 'official_tests_complete', 'input_mode', 'generated_checker', 'executable', 'generated_tests']
[Test set] Number of samples : 468
[Test set] Columns 27           : ['id', 'aliases', 'contest_id', 'contest_name', 'contest_type', 'contest_start', 'contest_start_year', 'index', 'time_limit', 'memory_limit', 'title', 'description', 'input_format', 'output_format', 'interaction_format', 'note', 'examples', 'editorial', 'rating', 'tags', 'testset_size', 'official_tests', 'official_tests_complete', 'input_mode', 'generated_checker', 'executable', 'generated_tests']


In [2]:

# # ============================================================
# # Dataset Features
# # ============================================================

# print("\n" + "=" * 70)
# print("Dataset Features")
# print("=" * 70)

# for column, feature in codeforces_test.features.items():
#     print(f"{column:30s}: {feature}")

In [3]:
# # ============================================================
# # Sample
# # ============================================================

# print("\n" + "=" * 70)
# print("Sample 0")
# print("=" * 70)

# sample = codeforces_test[0]

# for key, value in sample.items():
#     print(f"\n--- {key} ---")
#     print(value)

#### 필터링

In [4]:
# train
train_subset = codeforces_train.filter(
    lambda x:
        x["official_tests_complete"] is True
        and x["contest_start_year"] in [2023, 2024]
)

# test
test_subset = codeforces_test.filter(
    lambda x:
        x["official_tests_complete"] is True
        and x["contest_start_year"] in [2023, 2024]
)

In [5]:
# train 데이터 분석 코드
# import numpy as np
# from collections import Counter

# # official tests가 완전하게 제공되는 문제만 필터링
# cf_complete = codeforces_train.filter(
#     lambda x: x["official_tests_complete"] is True
# )

# print("=" * 70)
# print("Codeforces: Official Tests Complete")
# print("=" * 70)

# print(f"Original samples : {len(codeforces_train)}")
# print(f"Complete samples : {len(cf_complete)}")
# print(f"Removed samples  : {len(codeforces_train) - len(cf_complete)}")

# print(
#     f"Complete ratio   : "
#     f"{len(cf_complete) / len(codeforces_train) * 100:.1f}%"
# )

# print("\n" + "=" * 70)
# print("Contest Year Distribution")
# print("=" * 70)

# year_counts = Counter(
#     x["contest_start_year"]
#     for x in cf_complete
#     if x["contest_start_year"] is not None
# )

# for year, count in sorted(year_counts.items()):
#     percentage = count / len(cf_complete) * 100
#     print(f"{year}: {count:4d} ({percentage:5.1f}%)")

# diagnostic_cf = cf_complete.filter(
#     lambda x: x["contest_start_year"] in [2023, 2024]
# )

# print("\n" + "=" * 70)
# print("Contest Year Distribution")
# print("=" * 70)

# year_counts = Counter(
#     x["contest_start_year"]
#     for x in diagnostic_cf
#     if x["contest_start_year"] is not None
# )

# for year, count in sorted(year_counts.items()):
#     percentage = count / len(diagnostic_cf) * 100
#     print(f"{year}: {count:4d} ({percentage:5.1f}%)")
    

In [6]:
from datasets import concatenate_datasets
train_subset = train_subset.add_column(
    "source_split",
    ["train"] * len(train_subset)
)

test_subset = test_subset.add_column(
    "source_split",
    ["test"] * len(test_subset)
)
dataset = concatenate_datasets([
    train_subset,
    test_subset,
])

print(f"Total problems: {len(dataset)}")

Total problems: 98


In [7]:
# # id 기준 중복 확인
# ids = dataset["id"]

# print(f"Total problems      : {len(dataset)}")
# print(f"Unique problem IDs  : {len(set(ids))}")
# print(f"Duplicate problems  : {len(ids) - len(set(ids))}")

In [ ]:
# # 저장
# output_path = "/mnt/hdd/project_sLM_planning/data/codeforces"

# dataset.save_to_disk(output_path)

# print(f"Saved to: {output_path}")

Saving the dataset (1/1 shards): 100%|██████████| 98/98 [00:00<00:00, 749.77 examples/s]

Saved to: /mnt/hdd/project_sLM_planning/data/codeforces


In [9]:
from datasets import load_from_disk

dataset = load_from_disk(
    "/mnt/hdd/project_sLM_planning/data/codeforces"
)

print(f"Total problems: {len(dataset)}")
print(dataset.column_names)

Total problems: 98
['id', 'aliases', 'contest_id', 'contest_name', 'contest_type', 'contest_start', 'contest_start_year', 'index', 'time_limit', 'memory_limit', 'title', 'description', 'input_format', 'output_format', 'interaction_format', 'note', 'examples', 'editorial', 'rating', 'tags', 'testset_size', 'official_tests', 'official_tests_complete', 'input_mode', 'generated_checker', 'executable', 'generated_tests', 'source_split']


---

### 2. Rating Distribution

In [10]:
# ============================================================
# Rating Distribution
# ============================================================
from collections import Counter

ratings = []

for sample in dataset:
    rating = sample["rating"]

    if rating is not None:
        ratings.append(int(rating))


rating_counts = Counter(ratings)


print("=" * 70)
print("Rating Distribution")
print("=" * 70)

print(f"Total samples       : {len(dataset)}")
print(f"Samples with rating : {len(ratings)}")
print(f"Missing rating      : {len(dataset) - len(ratings)}")

print("\n" + "-" * 70)
print(f"{'Rating':>10} {'Count':>10} {'Percentage':>12}")
print("-" * 70)

for rating in sorted(rating_counts):
    count = rating_counts[rating]
    percentage = count / len(ratings) * 100

    print(
        f"{rating:>10} "
        f"{count:>10} "
        f"{percentage:>11.1f}%"
    )

print("-" * 70)

Rating Distribution
Total samples       : 98
Samples with rating : 85
Missing rating      : 13

----------------------------------------------------------------------
    Rating      Count   Percentage
----------------------------------------------------------------------
       800         20        23.5%
       900          2         2.4%
      1000          1         1.2%
      1100          2         2.4%
      1200          2         2.4%
      1300          1         1.2%
      1400          2         2.4%
      1600          2         2.4%
      1700          2         2.4%
      1900          1         1.2%
      2000          3         3.5%
      2100          3         3.5%
      2200          4         4.7%
      2300          4         4.7%
      2400          5         5.9%
      2500          3         3.5%
      2600          1         1.2%
      2700          8         9.4%
      2800          4         4.7%
      2900          2         2.4%
      3000          5      

In [11]:
print("\n" + "=" * 70)
print("Rating Statistics")
print("=" * 70)

print(f"Minimum : {min(ratings)}")
print(f"Maximum : {max(ratings)}")
print(f"Mean    : {sum(ratings) / len(ratings):.1f}")

sorted_ratings = sorted(ratings)

n = len(sorted_ratings)

if n % 2 == 0:
    median = (
        sorted_ratings[n // 2 - 1]
        + sorted_ratings[n // 2]
    ) / 2
else:
    median = sorted_ratings[n // 2]

print(f"Median  : {median}")


Rating Statistics
Minimum : 800
Maximum : 3500
Mean    : 1972.9
Median  : 2200


### 3. Tag Distribution

In [12]:
# ============================================================
# Tag Distribution
# ============================================================

tag_counter = Counter()
missing_tags = 0

for sample in dataset:
    tags = sample["tags"]

    if not tags:
        missing_tags += 1
        continue

    for tag in tags:
        tag_counter[tag] += 1


print("=" * 70)
print("Tag Distribution")
print("=" * 70)

print(f"Total samples       : {len(dataset)}")
print(f"Samples with tags   : {len(dataset) - missing_tags}")
print(f"Missing/empty tags  : {missing_tags}")
print(f"Unique tags         : {len(tag_counter)}")


print("\n" + "-" * 70)
print(f"{'Tag':<40} {'Count':>10} {'Percentage':>12}")
print("-" * 70)

for tag, count in tag_counter.most_common():
    percentage = count / len(dataset) * 100

    print(
        f"{tag:<40} "
        f"{count:>10} "
        f"{percentage:>11.1f}%"
    )

print("-" * 70)

Tag Distribution
Total samples       : 98
Samples with tags   : 97
Missing/empty tags  : 1
Unique tags         : 28

----------------------------------------------------------------------
Tag                                           Count   Percentage
----------------------------------------------------------------------
math                                             43        43.9%
brute force                                      37        37.8%
dp                                               33        33.7%
constructive algorithms                          29        29.6%
combinatorics                                    23        23.5%
implementation                                   22        22.4%
*special                                         19        19.4%
greedy                                           13        13.3%
number theory                                    11        11.2%
interactive                                       6         6.1%
strings                   

### 4. Rating × Tag

In [13]:
from collections import defaultdict
from statistics import mean, median

# ============================================================
# Rating × Tag
# ============================================================

tag_ratings = defaultdict(list)

for sample in dataset:

    rating = sample["rating"]
    tags = sample["tags"]

    if rating is None or not tags:
        continue

    rating = int(rating)

    for tag in tags:
        tag_ratings[tag].append(rating)


# ============================================================
# Statistics
# ============================================================

print("=" * 90)
print("Rating × Tag Statistics")
print("=" * 90)

print(
    f"{'Tag':<40}"
    f"{'Count':>8}"
    f"{'Mean':>10}"
    f"{'Median':>10}"
    f"{'Min':>8}"
    f"{'Max':>8}"
)

print("-" * 90)


# Tag count 기준으로 정렬
sorted_tags = sorted(
    tag_ratings.items(),
    key=lambda x: len(x[1]),
    reverse=True
)


for tag, ratings in sorted_tags:

    print(
        f"{tag:<40}"
        f"{len(ratings):>8}"
        f"{mean(ratings):>10.1f}"
        f"{median(ratings):>10.0f}"
        f"{min(ratings):>8}"
        f"{max(ratings):>8}"
    )

print("-" * 90)

Rating × Tag Statistics
Tag                                        Count      Mean    Median     Min     Max
------------------------------------------------------------------------------------------
math                                          40    1907.5      2050     800    3100
brute force                                   33    1981.8      2100     800    3500
dp                                            33    2478.8      2600    1100    3500
constructive algorithms                       26    1838.5      1700     800    3100
combinatorics                                 23    2417.4      2400     900    3100
implementation                                18    1438.9       800     800    2800
greedy                                        13    1992.3      2100     800    3000
number theory                                 10    1950.0      2200     800    3000
interactive                                    6    2700.0      2750    2100    3100
strings                            

In [14]:


# ============================================================
# Rating Bins
# ============================================================

RATING_BINS = [
    ("800-1199", 800, 1199),
    ("1200-1599", 1200, 1599),
    ("1600-1999", 1600, 1999),
    ("2000-2399", 2000, 2399),
    ("2400-2799", 2400, 2799),
    ("2800+", 2800, 9999),
]


def get_rating_bin(rating):
    for name, min_rating, max_rating in RATING_BINS:
        if min_rating <= rating <= max_rating:
            return name

    return None


# ============================================================
# Rating Bin × Tag
# ============================================================

bin_tag_counter = defaultdict(Counter)
bin_problem_count = Counter()

for sample in dataset:

    rating = sample["rating"]
    tags = sample["tags"]

    if rating is None or not tags:
        continue

    rating = int(rating)

    rating_bin = get_rating_bin(rating)

    if rating_bin is None:
        continue

    # 문제 수
    bin_problem_count[rating_bin] += 1

    # Tag별 카운트
    for tag in tags:
        bin_tag_counter[rating_bin][tag] += 1


# ============================================================
# Print Result
# ============================================================

print("=" * 100)
print("Rating Bin × Tag Distribution")
print("=" * 100)

for bin_name, _, _ in RATING_BINS:

    print(f"\n{'=' * 70}")
    print(f"Rating Bin: {bin_name}")
    print(f"Number of problems: {bin_problem_count[bin_name]}")
    print(f"{'=' * 70}")

    print(
        f"{'Tag':<40}"
        f"{'Count':>10}"
        f"{'Percentage':>12}"
    )

    print("-" * 70)

    total = bin_problem_count[bin_name]

    for tag, count in bin_tag_counter[bin_name].most_common():

        percentage = count / total * 100

        print(
            f"{tag:<40}"
            f"{count:>10}"
            f"{percentage:>11.1f}%"
        )

Rating Bin × Tag Distribution

Rating Bin: 800-1199
Number of problems: 25
Tag                                          Count  Percentage
----------------------------------------------------------------------
math                                            11       44.0%
implementation                                  10       40.0%
brute force                                      9       36.0%
constructive algorithms                          9       36.0%
number theory                                    3       12.0%
sortings                                         2        8.0%
strings                                          2        8.0%
games                                            2        8.0%
greedy                                           2        8.0%
*special                                         1        4.0%
dp                                               1        4.0%
combinatorics                                    1        4.0%

Rating Bin: 1200-1599
Number of pr